## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [ ]:
import statsmodels.stats.api as sms

# Вхідні параметри
p1 = 0.19  # Базове утримання (19%)
p2 = 0.20  # Очікуване утримання (19% + 1% ефекту)
alpha = 0.05
power = 0.80

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Обчислюємо розмір ефекту (Cohen's h)
effect_size = sms.proportion_effectsize(p1, p2)

# 2. Обчислюємо розмір вибірки для однієї групи
n_per_group = sms.NormalIndPower().solve_power(
    effect_size,
    power=power,
    alpha=alpha,
    ratio=1
)

# Округлюємо до більшого цілого та рахуємо загальну кількість
n_per_group = int(round(n_per_group))
total_n = n_per_group * 2

print(f"Користувачів у кожній групі: {n_per_group}")
print(f"Загальна кількість користувачів: {total_n}")


Користувачів у кожній групі: 24638
Загальна кількість користувачів: 49276


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [ ]:
!ls drive/MyDrive/Files/cookie_cats.csv
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

drive/MyDrive/Files/cookie_cats.csv


In [ ]:
df=pd.read_csv('drive/MyDrive/Files/cookie_cats.csv')

In [ ]:
# Розрахунок середнього значення retention_7 за версіями гри
retention_7_mean = df.groupby('version')['retention_7'].mean()
print(retention_7_mean)

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


на основі отриманих середніх значень, ми бачимо, що версія gate_30 демонструє вищий показник утримання порівняно з gate_40. Для перевірки того, чи є ця різниця випадковою, формулюємо наступні гіпотези:

* Нульова гіпотеза Hо: Переміщення воріт на рівень 40 не покращило або навіть погіршило утримання. Різниця між середніми показниками версій gate_30 та gate_40 статистично не значуща.\(\mu _gate_30 \mu _{gate\_40}\)
* Альтернативна гіпотеза H*a*: Версія з воротами на рівні 30 забезпечує статистично значуще вище утримання гравців через 7 днів після встановлення гри.\(\mu _{gate\_30}>\mu _{gate\_40}\)


**Тип статистичного тесту:** Правосторонній Z-тест для порівняння часток (proportions).

***Висновок:***

Згідно з розрахунками, версія gate_30 демонструє вищий рівень утримання гравців на 7-й день (19.02%) порівняно з версією gate_40 (18.20%). Отже, за попередніми даними, версія з воротами на 30-му рівні дає краще утримання.

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [ ]:
# 1. Підготовка даних
control = df[df['version'] == 'gate_30']['retention_7']
treatment = df[df['version'] == 'gate_40']['retention_7']

# Кількість успіхів (True) та загальна кількість спроб (n)
successes = np.array([control.sum(), treatment.sum()])
nobs = np.array([control.count(), treatment.count()])

# 2. Z-тест (двосторонній)
z_stat, p_val = proportions_ztest(count=successes, nobs=nobs, alternative='two-sided')

# 3. Довірчі інтервали для кожної групи (95%)
ci_control = proportion_confint(successes[0], nobs[0], alpha=0.05, method='normal')
ci_treatment = proportion_confint(successes[1], nobs[1], alpha=0.05, method='normal')

# 4. Виведення результатів
print(f"z statistic: {z_stat:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"Довірчий інтервал 95% для групи control: [{ci_control[0]:.4f}, {ci_control[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{ci_treatment[0]:.4f}, {ci_treatment[1]:.4f}]")


z statistic: 3.1644
p-value: 0.0016
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


**Висновок:**

Чи є статистично значущою різниця?

Так, різниця є статистично значущою. Оскільки p-value = 0.0016 значно менше за рівень значущості lpha =0.05, ми відхиляємо нульову гіпотезу. Це означає, що версія гри впливає на утримання гравців на 7-й день.

Чи перетинаються довірчі інтервали? Про що це каже?

Довірчі інтервали не перетинаються (інтервал для gate_30 знаходиться вище за інтервал для gate_40).
* gate_30: [0.1866, 0.1938]
* gate_40: [0.1785, 0.1855]

Це підсилює висновок про статистичну значущість: ми з 95% впевненістю можемо стверджувати, що справжнє значення утримання в контрольній групі вище, ніж у тестовій. Переміщення воріт на 40 рівень призвело до погіршення показника retention_7.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


**Формулювання гіпотез**

Для тесту Хі-квадрат на незалежність гіпотези формулюються наступним чином:
* Нульова гіпотеза Ho: Версія гри та утримання гравця на 7-й день незалежні. Це означає, що зміна рівня воріт не впливає на те, чи повернеться гравець у гру.
* Альтернативна гіпотеза H*a*: Між версією гри та утриманням гравця на 7-й день існує статистично значуща залежність.

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

# Створення таблиці спряженості
contingency_table = pd.crosstab(df['version'], df['retention_7'])

# Виконання тесту Хі-квадрат
chi2, p_val, dof, expected = chi2_contingency(contingency_table)

print(f"Таблиця спряженості:\n{contingency_table}\n")
print(f"p-value: {p_val:.4f}")


Таблиця спряженості:
retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279

p-value: 0.0016


**Результати та висновок**

 Виходячи з аналізу даних Cookie Cats:
 * p-value становить приблизно 0.0016 (може дещо варіюватися залежно від очищення даних, але стабільно < 0.05).
 * Висновок: Оскільки отримане p-value 0.0016 є значно меншим за встановлений рівень значущості alpha =0.05, ми відхиляємо нульову гіпотезу.

 **Загальний підсумок:**

 Ми маємо статистично значущі докази того, що існує залежність між рівнем, на якому встановлені ворота (30 або 40), та утриманням гравців на 7-й день. Оскільки показники у версії gate_30 вищі, це свідчить про те, що переміщення воріт на 40-й рівень негативно вплинуло на лояльність користувачів. Компанії рекомендується залишити ворота на 30-му рівні.